In [ ]:
import pandas as pd
import numpy as np

#Activation Function
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def loss_function(m, b, data):
    total_loss = 0
    N = len(data)
    for i in range(N):
        x = data[i, 0]
        y = data[i, 1]
        z = m * x + b
        y_pred = sigmoid(z)
        #Log-Loss (Binary Cross Entropy)
        total_loss += -(y * np.log(y_pred + 1e-15) + (1 - y) * np.log(1 - y_pred + 1e-15))
    return total_loss / N

def batch_gradient_descent(m, b, data, learning_rate):
    m_gradient = 0
    b_gradient = 0
    N = len(data)
    
    for i in range(N):
        x = data[i, 0]
        y = data[i, 1]
        y_pred = sigmoid(m * x + b)
        
        #Update for batch
        m_gradient += (y - y_pred) * x
        b_gradient += (y - y_pred)
    
    m -= learning_rate * (m_gradient / N)
    b -= learning_rate * (b_gradient / N)
    
    return m, b

def stochastic_gradient_descent(m, b, data, learning_rate):
    N = len(data)
    for i in range(N):
        x = data[i, 0]
        y = data[i, 1]
        y_pred = sigmoid(m * x + b)
        
        #Update for each sample
        m_grad = (y - y_pred) * x
        b_grad = (y - y_pred)
        
        m -= learning_rate * m_grad
        b -= learning_rate * b_grad
    
    return m, b

def mini_batch_gradient_descent(m,b,data,learning_rate, batch_size):

    N = len(data)
    for i in range(0, N, batch_size): #loop through mini-batches
        m_gradient = 0
        b_gradient = 0
        batch = data[i:i+batch_size]
        k = len(batch)
        for j in range(k): #loop through each data point in the mini-batch
            x = batch[j,0]
            y = batch[j,1]
            y_pred = sigmoid(m * x + b)
            m_gradient += -(2/N) * x * (y - y_pred) #partial derivative with respect to m
            b_gradient += -(2/N) * (y - y_pred) #partial derivative with respect to b
        m -= learning_rate * m_gradient #update m
        b -= learning_rate * b_gradient #update b
    return m, b

def train(data, learning_rate=0.1, epochs=1000, method='batch'):
    m = 0.0
    b = 0.0
    
    for epoch in range(epochs):
        if method == 'batch':
            m, b = batch_gradient_descent(m, b, data, learning_rate)
        elif method == 'stochastic':
            m, b = stochastic_gradient_descent(m, b, data, learning_rate)
        
        if epoch % 100 == 0:
            print(f'Epoch {epoch}, Loss: {loss_function(m, b, data):.4f}')
    
    return m, b

if __name__ == "__main__":
    #Data
    data = np.array([
        [0.5, 0],
        [1.0, 0],
        [2.5, 1],
        [3.0, 1],
        [4.0, 1]
    ])
    
    m, b = train(data, learning_rate=0.1, epochs=1000, method='batch')
    print(f'\nParâmetros finais: m = {m:.4f}, b = {b:.4f}')
    
    #Test
    teste_x = 2.0
    prob = sigmoid(m * teste_x + b)
    print(f'Probabilidade para x={teste_x}: {prob:.2%}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

#Data
np.random.seed(42)
X = np.random.randn(100, 1) * 2  #Features
#Label
prob_real = 1 / (1 + np.exp(-(2.5 * X - 1))) 
y = (np.random.rand(100, 1) < prob_real).astype(int)
data = np.hstack((X, y))

# --- Functions ---

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def loss_function(m, b, data):
    X, y = data[:, 0], data[:, 1]
    y_pred = sigmoid(m * X + b)
    # Log-Loss (with epsilon to avoid log(0))
    eps = 1e-15
    return -np.mean(y * np.log(y_pred + eps) + (1 - y) * np.log(1 - y_pred + eps))

def batch_gradient_descent(m, b, data, lr):
    X, y = data[:, 0], data[:, 1]
    N = len(data)
    y_pred = sigmoid(m * X + b)
    error = y_pred - y
    #Log-Loss Gradient
    m_grad = np.mean(error * X)
    b_grad = np.mean(error)
    return m - lr * m_grad, b - lr * b_grad

def stochastic_gradient_descent(m, b, data, lr):
    idx = np.random.randint(len(data))
    xi, yi = data[idx, 0], data[idx, 1]
    y_pred = sigmoid(m * xi + b)
    error = y_pred - yi
    m_grad = error * xi
    b_grad = error
    return m - lr * m_grad, b - lr * b_grad

def train(data, learning_rate=0.1, epochs=100, method='batch'):
    m, b = -2.0, 0.0 #Starting point
    theta_history = []
    cost_history = []
    
    for epoch in range(epochs):
        if method == 'batch':
            m, b = batch_gradient_descent(m, b, data, learning_rate)
        elif method == 'stochastic':
            m, b = stochastic_gradient_descent(m, b, data, learning_rate)
        
        theta_history.append((m, b))
        cost_history.append(loss_function(m, b, data))
            
    return np.array(theta_history), np.array(cost_history)

#Execution
metodo_escolhido = 'batch' 
n_iterations = 100
t_history, c_history = train(data, learning_rate=0.5, epochs=n_iterations, method=metodo_escolhido)

# --- Setting up the visualization ---
m_range = np.linspace(-1, 5, 50)
b_range = np.linspace(-3, 3, 50)
M, B = np.meshgrid(m_range, b_range)
Z = np.array([loss_function(mi, bi, data) for mi, bi in zip(np.ravel(M), np.ravel(B))]).reshape(M.shape)

plt.ioff()
fig = plt.figure(figsize=(18, 6))
ax1 = fig.add_subplot(131) #Sigmoid Fit
ax2 = fig.add_subplot(132) #Log-Loss
ax3 = fig.add_subplot(133, projection='3d') #Error Surface

def animate(i):
    ax1.clear(); ax2.clear(); ax3.clear()
    curr_m, curr_b = t_history[i]
    
    #Plot 1: Sigmoid Fit
    ax1.scatter(X, y, alpha=0.5, c=y, cmap='bwr')
    x_line = np.linspace(X.min(), X.max(), 100)
    y_line = sigmoid(curr_m * x_line + curr_b)
    ax1.plot(x_line, y_line, color='green', lw=3)
    ax1.axhline(0.5, color='black', linestyle='--', alpha=0.3)
    ax1.set_title(f"Logistic Fit - Iter: {i}\nProb(y=1 | x)")

    # Plot 2: Cost Evolution (Log-Loss)
    ax2.plot(range(i+1), c_history[:i+1], color='green')
    ax2.set_xlim(0, n_iterations); ax2.set_ylim(0, max(c_history) * 1.1)
    ax2.set_ylabel("Log-Loss")

    # Plot 3: Parameter Space
    ax3.plot_surface(M, B, Z, cmap='viridis', alpha=0.3)
    ax3.plot(t_history[:i+1, 0], t_history[:i+1, 1], c_history[:i+1], color='red', lw=2, marker="o", markersize=3)
    ax3.set_title("Path in Parameter Space")
    ax3.view_init(elev=35, azim=-120)

ani = FuncAnimation(fig, animate, frames=n_iterations, interval=100)
HTML(ani.to_jshtml())

In [ ]:
ani.save('gradiente_descendente.gif', writer='pillow', fps=15) #saving the animation as a GIF